In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tree.xg_classifier import XGClassifier
from xgboost import XGBClassifier, plot_tree
import shap
from common.helper import load_data_and_scale_pos_weight
from mini_shap.independent_tree_shap import IndependentTreeExplainer
from tree.xg_classifier import XGClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import auc, precision_recall_curve

/home/sakane/Documents/projects/python/mini_xgshap/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
both_bin_path = "../datasets/BRFSS/feat_eng/both_bin.parquet"
both_grp_path = "../datasets/BRFSS/feat_eng/both_grp.parquet"
both_og_path = "../datasets/BRFSS/feat_eng/both_og.parquet"

both_bin_model_path = "../datasets/BRFSS/feat_eng/models/both_bin_model.ubj"
both_grp_model_path = "../datasets/BRFSS/feat_eng/models/both_grp_model.ubj"
both_og_model_path = "../datasets/BRFSS/feat_eng/models/both_og_model.ubj"

In [3]:
X_both_og_train, X_both_og_test, y_both_og_train, y_both_og_test, scale_pos_weight = load_data_and_scale_pos_weight(cast_string=True, dataset_path=both_og_path)
X_both_og_train_np, X_both_og_test_np, y_both_og_train_np, y_both_og_test_np = X_both_og_train.to_numpy().astype(np.float32), X_both_og_test.to_numpy().astype(np.float32), y_both_og_train.to_numpy().astype(np.float32), y_both_og_test.to_numpy().astype(np.float32)

X_both_grp_train, X_both_grp_test, y_both_grp_train, y_both_grp_test, scale_pos_weight = load_data_and_scale_pos_weight(cast_string=True, dataset_path=both_grp_path)
X_both_grp_train_np, X_both_grp_test_np, y_both_grp_train_np, y_both_grp_test_np = X_both_grp_train.to_numpy().astype(np.float32), X_both_grp_test.to_numpy().astype(np.float32), y_both_grp_train.to_numpy().astype(np.float32), y_both_grp_test.to_numpy().astype(np.float32)

X_both_bin_train, X_both_bin_test, y_both_bin_train, y_both_bin_test, scale_pos_weight = load_data_and_scale_pos_weight(cast_string=True, dataset_path=both_bin_path)
X_both_bin_train_np, X_both_bin_test_np, y_both_bin_train_np, y_both_bin_test_np = X_both_bin_train.to_numpy().astype(np.float32), X_both_bin_test.to_numpy().astype(np.float32), y_both_bin_train.to_numpy().astype(np.float32), y_both_bin_test.to_numpy().astype(np.float32)


In [4]:
both_og_model = XGBClassifier()
both_og_model.load_model(both_og_model_path)
both_og_model2 = XGClassifier.load_from_xgbclassifier(both_og_model, columns=list(X_both_og_train.columns))

In [5]:
both_bin_model = XGBClassifier()
both_bin_model.load_model(both_bin_model_path)
both_bin_model2 = XGClassifier.load_from_xgbclassifier(both_bin_model, columns=list(X_both_bin_train.columns))

In [6]:
both_grp_model = XGBClassifier()
both_grp_model.load_model(both_grp_model_path)
both_grp_model2 = XGClassifier.load_from_xgbclassifier(both_grp_model, columns=list(X_both_grp_train.columns))

In [7]:
indices = np.random.choice(len(X_both_og_train_np), 1000, replace=False)

background_both_og_sample = X_both_og_train_np[indices]
foreground_both_og_sample = X_both_og_test_np

background_both_bin_sample = X_both_bin_train_np[indices]
foreground_both_bin_sample = X_both_bin_test_np

background_both_grp_sample = X_both_grp_train_np[indices]
foreground_both_grp_sample = X_both_grp_test_np

In [8]:
expl_both_og = IndependentTreeExplainer(
    model=both_og_model2,
    refset=background_both_og_sample,
    model_output='raw',
)

shap_values_both_og = expl_both_og.full_vectorized_explain(foreground_both_og_sample)

In [9]:
expl_both_bin = IndependentTreeExplainer(
    model=both_bin_model2,
    refset=background_both_bin_sample,
    model_output='raw',
)

shap_values_both_bin = expl_both_bin.full_vectorized_explain(foreground_both_bin_sample)

In [10]:
expl_both_grp = IndependentTreeExplainer(
    model=both_grp_model2,
    refset=background_both_grp_sample,
    model_output='raw',
)

shap_values_both_grp = expl_both_grp.full_vectorized_explain(foreground_both_grp_sample)

In [12]:
expl_both_og.export_shap_values("../datasets/BRFSS/feat_eng/shap_values/expl_both_og_shap_values.npy", shap_values_both_og)
expl_both_bin.export_shap_values("../datasets/BRFSS/feat_eng/shap_values/expl_both_bin_shap_values.npy", shap_values_both_bin)
expl_both_grp.export_shap_values("../datasets/BRFSS/feat_eng/shap_values/expl_both_grp_shap_values.npy", shap_values_both_grp)